# ✅ THEORY 
##### KNN Imputer = Missing value fill method based on "Nearest Neighbors"
#####
#### How it works?
##### - For a row where Age is missing,
#####   it finds K most similar rows using other available features
#####   (here: Pclass and Fare).
##### - Then it fills missing Age using the average of those neighbors.
#####
#### Important point:
##### - KNN uses "distance" (like Euclidean distance).
##### - If features are on different scales (Fare is big, Pclass is small),
#####   distance becomes biased.
##### ✅ That’s why scaling BEFORE KNNImputer is very important.
#####
#### When to use KNNImputer?
##### ✅ Good when missing values depend on other columns (pattern exists)
##### ✅ Often better than mean/median imputation
#####
#### When NOT to use?
##### ❌ Very large datasets (slow)
##### ❌ Too many missing values (neighbors not reliable)

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("train.csv")[["Age", "Pclass", "Fare", "Survived"]]

print("✅ Data sample:\n", df.head(), "\n")
print("✅ Missing %:\n", df.isnull().mean() * 100, "\n")

# Features & target
X = df.drop(columns=["Survived"])
y = df["Survived"]


✅ Data sample:
     Age  Pclass     Fare  Survived
0  22.0       3   7.2500         0
1  38.0       1  71.2833         1
2  26.0       3   7.9250         1
3  35.0       1  53.1000         1
4  35.0       3   8.0500         0 

✅ Missing %:
 Age         19.86532
Pclass       0.00000
Fare         0.00000
Survived     0.00000
dtype: float64 



In [4]:
# Step 2) Train-test split (stratify keeps class balance same)
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

In [5]:
# Step 3) ✅ KNN Imputation Pipeline
# (Scaler -> KNNImputer -> LogisticRegression)
# ==========================================================
knn_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(n_neighbors=3, weights="distance")),
    ("model", LogisticRegression(max_iter=2000))
])

knn_pipeline.fit(X_train, y_train)
y_pred_knn = knn_pipeline.predict(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
print("✅ Accuracy WITH KNNImputer:", acc_knn)


✅ Accuracy WITH KNNImputer: 0.6927374301675978


In [6]:
# ==========================================================
# Step 4) ✅ Baseline Pipeline (SimpleImputer mean)
# (Imputer -> LogisticRegression)
#
# SimpleImputer mean = replaces missing Age with mean Age of training set
# This is fast + simple but ignores relationships between columns.
# ==========================================================
simple_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", LogisticRegression(max_iter=2000))
])

simple_pipeline.fit(X_train, y_train)
y_pred_simple = simple_pipeline.predict(X_test)

acc_simple = accuracy_score(y_test, y_pred_simple)
print("✅ Accuracy WITH SimpleImputer(mean):", acc_simple)


✅ Accuracy WITH SimpleImputer(mean): 0.6871508379888268


In [7]:
# ==========================================================
# Step 5) Final Comparison
# ==========================================================
print("\n📌 Final Comparison:")
print(f"➡️ KNNImputer Accuracy       : {acc_knn:.4f}")
print(f"➡️ SimpleImputer(mean) Accuracy: {acc_simple:.4f}")


📌 Final Comparison:
➡️ KNNImputer Accuracy       : 0.6927
➡️ SimpleImputer(mean) Accuracy: 0.6872
